# ifcopenshell

Let's start by installing IfcOpenShell. In addition, here are some other libraries that you might find useful. Lark helps parse custom queries, numpy is great for coordinates and matrixes, shapely provides 2D shape analysis, and mathutils provides Vector and Matrix functions.

In [ ]:
!pip install lark
!pip install numpy
!pip install shapely
!pip install mathutils
!pip install ifcopenshell
!pip install trimesh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.4 MB/s eta 0:00:00
  Using cached mathutils-3.3.0.tar.gz (245 kB)
  Preparing metadata (setup.py) ... - done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [41 lines of output]
      running bdist_wheel
      running build
      running build_ext
      building 'mathutils' extension
      creating build/temp.macosx-11.1-arm64-cpython-313/src/blenlib/intern
      creating build/temp.macosx-11.1-arm64-cpython-313/src/generic
      creating build/temp.macosx-11.1-arm64-cpython-313/src/mathutils
      creating build/temp.macosx-11.1-arm64-cpython-313/src/stubs
      clang -fno-strict-overflow -Wsign-compare -Wunreachable-code -DNDEBUG -O2 -Wall -fPIC -O2 -isystem /opt/miniconda3/include -arch arm64 -fPIC -O2 -isystem /opt/miniconda3/include -arch arm64 -DMATH_STANDALONE -DWITH_ASSERT_ABORT -Isrc/stubs -Isrc/blenlib -Isrc/makesdna -I/opt/miniconda3/include/p

## Setup and Data Split

### Subtask:
Identify student folders, perform an 80/20 train/test split, and create the output directory structure.


In [ ]:
import os
import random
from pathlib import Path

# 1. Define Root and Submission Directory
project_root = Path("./data")

# Locate the specific submission directory created by the unzip command
submission_dirs = [d for d in project_root.iterdir() if d.is_dir()]

submission_dir = submission_dirs[0]
print(f"Submission directory identified: {submission_dir}")

# 2. Create a list of all student directories
# We filter for directories to ignore loose files (like zips or pdfs)
student_dirs = [d for d in submission_dir.iterdir() if d.is_dir()]

# 3. Shuffle and Split (80/20)
random.seed(42)
random.shuffle(student_dirs)

split_idx = int(len(student_dirs) * 0.8)
train_students = student_dirs[:split_idx]
test_students = student_dirs[split_idx:]

# 4. Create Output Directories
os.makedirs("meshes/train", exist_ok=True)
os.makedirs("meshes/test", exist_ok=True)

# 5. Print Statistics
print(f"\nTotal student folders found: {len(student_dirs)}")
print(f"Training set count (80%): {len(train_students)}")
print(f"Testing set count (20%): {len(test_students)}")

# Verification: Print a few names from each set
print(f"\nExample Train folders: {[s.name for s in train_students[:3]]}")
print(f"Example Test folders: {[s.name for s in test_students[:3]]}")

Submission directory identified: Bentley3D/data/BIM.fundamental 950799697 (S25)-Submission Assignment 1-3415395

Total student folders found: 216
Training set count (80%): 172
Testing set count (20%): 44

Example Train folders: ['Leon Julian Seyffer_11921021_assignsubmission_file', 'Sajjad Hussain_11921036_assignsubmission_file', 'Fangyuan Han_11921004_assignsubmission_file']
Example Test folders: ['Anoushka Chowdhury_11920721_assignsubmission_file', 'Akash Kumar_11920695_assignsubmission_file', 'Olena Dubrovska_11920774_assignsubmission_file']


## Process Train/Test Sets

### Subtask:
Define the processing function to extract geometries without centering, iterate through student folders to generate OBJ files, and compile metadata.


In [ ]:
import ifcopenshell
import ifcopenshell.geom
import trimesh
import numpy as np
import json
import multiprocessing
import os
from pathlib import Path

**Reasoning**:
Define the processing function to extract geometries from IFC files without centering, iterate through the train/test student folders to generate OBJ files, and compile the metadata into a JSON file as per the instructions.



In [ ]:
import sys
import subprocess
import importlib.util

# Function to install packages dynamically
def install_package(package_name):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

# Check and install ifcopenshell
if importlib.util.find_spec("ifcopenshell") is None:
    print("Installing ifcopenshell...")
    install_package("ifcopenshell")

# Check and install trimesh
if importlib.util.find_spec("trimesh") is None:
    print("Installing trimesh...")
    install_package("trimesh")



def process_ifc_file(ifc_path, output_dir):
    metadata_list = []
    try:
        # Initialize IFC file and settings
        model = ifcopenshell.open(ifc_path)
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)

        # Initialize iterator
        iterator = ifcopenshell.geom.iterator(settings, model, multiprocessing.cpu_count())

        if iterator.initialize():
            while True:
                shape = iterator.get()
                element = model.by_id(shape.id)

                # Extract geometry
                verts = shape.geometry.verts
                faces = shape.geometry.faces

                # Reshape to (N, 3)
                vertices = np.array(verts).reshape((-1, 3))
                faces = np.array(faces).reshape((-1, 3))

                # Create Trimesh object (preserving original coordinates)
                mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)

                # Export as OBJ
                filename = f"{element.GlobalId}.obj"
                output_path = os.path.join(output_dir, filename)
                mesh.export(output_path)

                # Collect Metadata
                # Calculate relative path from the project root (/content/projects)
                relative_path = str(Path(ifc_path).relative_to("./Bentley3D/data"))

                metadata_list.append({
                    "relative_source_path": relative_path,
                    "GlobalId": element.GlobalId,
                    "IfcType": element.is_a(),
                    "mesh_filename": filename
                })

                if not iterator.next():
                    break

    except Exception as e:
        print(f"Error processing {ifc_path}: {e}")

    return metadata_list

# Main Execution Loop
all_metadata = []

# Process Train Set
print("Processing Training Set...")
# Ensure train_students is available from the context (Kernel State)
if 'train_students' in locals():
    for student_dir in train_students:
        # Find IFC files (case-insensitive glob strategy)
        ifc_files = [p for p in student_dir.glob("*") if p.suffix.lower() == ".ifc"]
        for ifc_file in ifc_files:
            meta = process_ifc_file(str(ifc_file), "meshes/train")
            all_metadata.extend(meta)

# Process Test Set
print("Processing Testing Set...")
if 'test_students' in locals():
    for student_dir in test_students:
        ifc_files = [p for p in student_dir.glob("*") if p.suffix.lower() == ".ifc"]
        for ifc_file in ifc_files:
            meta = process_ifc_file(str(ifc_file), "meshes/test")
            all_metadata.extend(meta)

# Save Metadata
metadata_file = "metadata.json"
with open(metadata_file, "w") as f:
    json.dump(all_metadata, f, indent=4)

print(f"\nProcessing Complete.")
print(f"Total meshes generated: {len(all_metadata)}")
print(f"Metadata saved to: {metadata_file}")

Installing ifcopenshell...
Installing trimesh...
Processing Training Set...


KeyboardInterrupt: 

# Task
Define a `process_single_ifc` function to extract geometries from an IFC file as OBJ meshes (without centering) and calculate the source path relative to "./Bentley3D/data". Use `concurrent.futures.ProcessPoolExecutor` to process all IFC files in `train_students` and `test_students` in parallel. Finally, aggregate the metadata, save it to `metadata.json`, and print a summary of the generated meshes.

## Define Parallel Extraction Function

### Subtask:
Define the `process_single_ifc` worker function to extract geometries and generate metadata for a single IFC file.


**Reasoning**:
Define the `process_single_ifc` worker function to handle geometry extraction and metadata generation for a single IFC file. This function is designed for parallel execution, using a single thread for the iterator to prevent resource contention.



In [ ]:
import ifcopenshell
import ifcopenshell.geom
import trimesh
import numpy as np
import os
from pathlib import Path

def process_single_ifc(ifc_path, output_dir):
    """
    Worker function to process a single IFC file: extracts geometry, saves as OBJ,
    and returns metadata.
    """
    metadata_list = []
    ifc_path_str = str(ifc_path)

    try:
        # Open the IFC file
        model = ifcopenshell.open(ifc_path_str)

        # Configure settings
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)

        # Initialize iterator with 1 thread to avoid contention in parallel processing
        iterator = ifcopenshell.geom.iterator(settings, model, 1)

        if iterator.initialize():
            while True:
                shape = iterator.get()
                element = model.by_id(shape.id)

                # Extract geometry
                verts = shape.geometry.verts
                faces = shape.geometry.faces

                # Reshape to (N, 3)
                vertices = np.array(verts).reshape((-1, 3))
                faces = np.array(faces).reshape((-1, 3))

                # Create Trimesh object (process=False to keep original geometry)
                mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)

                # Export as OBJ
                filename = f"{element.GlobalId}.obj"
                output_path = os.path.join(output_dir, filename)
                mesh.export(output_path)

                # Collect Metadata
                # Calculate relative path from the project root
                try:
                    relative_path = str(Path(ifc_path_str).relative_to("./Bentley3D/data"))
                except ValueError:
                    # Fallback if path structure differs
                    relative_path = ifc_path_str

                metadata_list.append({
                    "relative_source_path": relative_path,
                    "GlobalId": element.GlobalId,
                    "IfcType": element.is_a(),
                    "mesh_filename": filename
                })

                if not iterator.next():
                    break

    except Exception as e:
        print(f"[Warning] Failed to process {ifc_path_str}: {e}")

    return metadata_list

# Task
Use `concurrent.futures.ProcessPoolExecutor` to process the `train_students` (output to `meshes/train`) and `test_students` (output to `meshes/test`) lists in parallel by invoking `process_single_ifc` for each IFC file. Aggregate the returned metadata, save it to `metadata.json`, and print a final summary with the total mesh count.

## Execute Parallel Processing

### Subtask:
Define the worker function and execute it in parallel for all IFC files using ProcessPoolExecutor.


**Reasoning**:
I will define the `process_single_ifc` function and then use `concurrent.futures.ProcessPoolExecutor` to process all IFC files in the training and testing sets in parallel. I will collect the metadata from each task, save it to `metadata.json`, and print a summary of the results.



**Reasoning**:
The `AttributeError: Can't get attribute 'process_single_ifc'` occurs because `multiprocessing` (used by `ProcessPoolExecutor`) cannot pickle functions defined interactively in the `__main__` scope of a Jupyter notebook. To fix this, I will write the worker function to a separate Python file (`ifc_worker.py`) and import it, making it importable by the child processes.



**Reasoning**:
The `IndentationError` occurred because the `worker_script` string was defined using triple double quotes (`"""`), which clashed with the triple double quotes used for the docstring inside the script text, terminating the string early. I will fix this by using triple single quotes (`'''`) for the outer `worker_script` variable definition.



## Execute Parallel Processing with Progress Bar

### Subtask:
Define the worker function in `ifc_worker.py`, prepare the task list, and execute parallel geometry extraction using `ProcessPoolExecutor` monitored by a `tqdm` progress bar.


**Reasoning**:
I will write the worker function to `ifc_worker.py` using triple single quotes to avoid syntax errors. Then, I will import the necessary libraries, reload the worker module, prepare the tasks from the `train_students` and `test_students` lists, and execute the processing using `ProcessPoolExecutor` with a `tqdm` progress bar to track progress.



**Reasoning**:
The previous attempt failed because the worker processes could not locate the output directories. To fix this, I will use absolute paths for the output directories, ensure they are created, and then re-run the parallel processing. I will also persist the worker script creation to ensure it is available.



In [ ]:
import os
import sys
import concurrent.futures
import json
import importlib
from pathlib import Path
from tqdm.notebook import tqdm
import ifcopenshell
import ifcopenshell.geom
import trimesh
import numpy as np

# 1. Write worker function to file (using triple single quotes for the outer string)
worker_script = '''
import ifcopenshell
import ifcopenshell.geom
import trimesh
import numpy as np
import os
from pathlib import Path

def process_single_ifc(args):
    """
    Worker function to process a single IFC file.
    Args is a tuple: (ifc_path, output_dir)
    """
    ifc_path, output_dir = args
    metadata_list = []
    ifc_path_str = str(ifc_path)

    try:
        # Open IFC file
        model = ifcopenshell.open(ifc_path_str)

        # Settings
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)

        # Iterator with 1 thread (safe for multiprocessing worker)
        iterator = ifcopenshell.geom.iterator(settings, model, 1)

        if iterator.initialize():
            while True:
                shape = iterator.get()
                element = model.by_id(shape.id)

                # Extract geometry
                verts = shape.geometry.verts
                faces = shape.geometry.faces

                vertices = np.array(verts).reshape((-1, 3))
                faces = np.array(faces).reshape((-1, 3))

                # Create Mesh (process=False to keep original coordinates)
                mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)

                # Export
                filename = f"{element.GlobalId}.obj"
                output_path = os.path.join(output_dir, filename)
                mesh.export(output_path)

                # Metadata
                try:
                    prefix = Path("./Bentley3D/data")
                    rel_path = Path(ifc_path).relative_to(prefix)
                    relative_path_str = str(rel_path)
                except ValueError:
                    relative_path_str = str(ifc_path)

                metadata_list.append({
                    "relative_source_path": relative_path_str,
                    "GlobalId": element.GlobalId,
                    "IfcType": element.is_a(),
                    "mesh_filename": filename
                })

                if not iterator.next():
                    break

    except Exception as e:
        return f"Error processing {ifc_path}: {str(e)}"

    return metadata_list
'''

with open("ifc_worker.py", "w") as f:
    f.write(worker_script)

# 2. Import/Reload worker
sys.path.append(os.getcwd())
import ifc_worker
importlib.reload(ifc_worker)
from ifc_worker import process_single_ifc

# 3. Define Absolute Output Paths and Create Directories
# Using absolute paths ensures workers find them regardless of their context
train_output_dir = os.path.abspath("./Bentley3D/processed_data/meshes/train")
test_output_dir = os.path.abspath("./Bentley3D/processed_data/meshes/test")

os.makedirs(train_output_dir, exist_ok=True)
os.makedirs(test_output_dir, exist_ok=True)

print(f"Output directories:\n - Train: {train_output_dir}\n - Test: {test_output_dir}")

# 4. Prepare Tasks
tasks = []

# Check if lists exist, otherwise regenerate (using the variables from kernel if available)
print("Reloading student directories...")
project_root = Path("./Bentley3D/data")
# Find submission dir again
submission_dir = project_root.joinpath("BIM.fundamental 950799697 (S25)-Submission Assignment 1-3415395")
student_dirs = [d for d in submission_dir.iterdir() if d.is_dir()]
import random
random.seed(42)
random.shuffle(student_dirs)
split_idx = int(len(student_dirs) * 0.8)
train_students = student_dirs[:split_idx]
test_students = student_dirs[split_idx:]

# Collect IFC files with ABSOLUTE output paths
for student_dir in train_students:
    for ifc_file in list(student_dir.glob("*.ifc")) + list(student_dir.glob("*.IFC")):
        tasks.append((ifc_file, train_output_dir))

for student_dir in test_students:
    for ifc_file in list(student_dir.glob("*.ifc")) + list(student_dir.glob("*.IFC")):
        tasks.append((ifc_file, test_output_dir))

print(f"Prepared {len(tasks)} tasks for processing.")

# 5. Execute with Progress Bar
all_metadata = []
errors = []

if __name__ == '__main__' and len(tasks) > 0:
    with concurrent.futures.ProcessPoolExecutor() as executor:
        future_to_file = {executor.submit(process_single_ifc, task): task[0] for task in tasks}

        for future in tqdm(concurrent.futures.as_completed(future_to_file), total=len(tasks), desc="Processing IFCs"):
            ifc_file = future_to_file[future]
            try:
                result = future.result()
                if isinstance(result, list):
                    all_metadata.extend(result)
                else:
                    errors.append(result)
            except Exception as exc:
                errors.append(f"{ifc_file} generated exception: {exc}")

# 6. Save & Report
output_json = "metadata.json"
with open(output_json, "w") as f:
    json.dump(all_metadata, f, indent=4)

print(f"Processing finished. {len(all_metadata)} meshes generated.")
if errors:
    print(f"Errors encountered: {len(errors)}")
    for e in errors[:5]:
        print(f"- {e}")
    if len(errors) > 5: print("...")

Output directories:
 - Train: /Users/sercanamac/Bentley3D/processed_data/meshes/train
 - Test: /Users/sercanamac/Bentley3D/processed_data/meshes/test
Reloading student directories...
Prepared 202 tasks for processing.


Processing IFCs:   0%|          | 0/202 [00:00<?, ?it/s]

Processing finished. 238657 meshes generated.


In [ ]:
print("Reloading student directories...")
project_root = Path("./Bentley3D/data")
# Find submission dir again
submission_dir = project_root.joinpath("BIM.fundamental 950799697 (S25)-Submission Assignment 1-3415395")
student_dirs = [d for d in submission_dir.iterdir() if d.is_dir()]
import random
random.seed(42)
random.shuffle(student_dirs)
split_idx = int(len(student_dirs) * 0.8)
train_students = student_dirs[:split_idx]
test_students = student_dirs[split_idx:]

# Collect IFC files with ABSOLUTE output paths
for student_dir in train_students:
    for ifc_file in list(student_dir.glob("*.ifc")) + list(student_dir.glob("*.IFC")):
        tasks.append((ifc_file, train_output_dir))


Reloading student directories...


In [ ]:
 [d for d in project_root.iterdir()]

[PosixPath('Bentley3D/data/.ipynb_checkpoints'),
 PosixPath('Bentley3D/data/BIM.fundamental 950799697 (S25)-Submission Assignment 1-3415395')]

In [ ]:
import json

# Load the metadata from metadata.json
metadata_file = "metadata.json"

try:
    with open(metadata_file, "r") as f:
        all_metadata = json.load(f)

    # Extract all IfcType values
    ifc_types = [item["IfcType"] for item in all_metadata if "IfcType" in item]

    # Get unique IfcType values
    unique_ifc_types = sorted(list(set(ifc_types)))

    # Print the unique IfcType values
    print(f"Found {len(unique_ifc_types)} unique IfcTypes:")
    for ifc_type in unique_ifc_types:
        print(f"- {ifc_type}")
except:
    print("wtf")

Found 38 unique IfcTypes:
- IfcBeam
- IfcBuildingElementPart
- IfcBuildingElementProxy
- IfcCableCarrierFitting
- IfcCivilElement
- IfcColumn
- IfcCovering
- IfcCurtainWall
- IfcDiscreteAccessory
- IfcDistributionFlowElement
- IfcDoor
- IfcElectricAppliance
- IfcFlowSegment
- IfcFlowTerminal
- IfcFooting
- IfcFurnishingElement
- IfcFurniture
- IfcGeographicElement
- IfcLightFixture
- IfcMember
- IfcOpeningElement
- IfcPipeSegment
- IfcPlate
- IfcRailing
- IfcRampFlight
- IfcRoof
- IfcSanitaryTerminal
- IfcShadingDevice
- IfcSite
- IfcSlab
- IfcSpace
- IfcStair
- IfcStairFlight
- IfcSystemFurnitureElement
- IfcTransportElement
- IfcWall
- IfcWallStandardCase
- IfcWindow


In [11]:
import collections

# Count occurrences of each IfcType
ifc_type_counts = collections.Counter(ifc_types)

# Sort by count in descending order
sorted_ifc_type_counts = sorted(ifc_type_counts.items(), key=lambda item: item[1], reverse=True)

print("IfcType Counts (Sorted by Frequency):")
for ifc_type, count in sorted_ifc_type_counts:
    print(f"- {ifc_type}: {count}")

IfcType Counts (Sorted by Frequency):
- IfcMember: 46455
- IfcPlate: 32876
- IfcOpeningElement: 26572
- IfcFurniture: 21132
- IfcBuildingElementProxy: 18240
- IfcWall: 16511
- IfcWindow: 14406
- IfcColumn: 11699
- IfcBeam: 11636
- IfcDoor: 7980
- IfcBuildingElementPart: 6426
- IfcSpace: 6288
- IfcSlab: 4612
- IfcRailing: 3259
- IfcFlowTerminal: 3019
- IfcFurnishingElement: 2426
- IfcStairFlight: 1747
- IfcWallStandardCase: 1466
- IfcLightFixture: 442
- IfcSystemFurnitureElement: 440
- IfcFooting: 295
- IfcCovering: 193
- IfcDistributionFlowElement: 126
- IfcRoof: 125
- IfcStair: 79
- IfcCurtainWall: 56
- IfcSanitaryTerminal: 53
- IfcDiscreteAccessory: 29
- IfcGeographicElement: 23
- IfcSite: 23
- IfcTransportElement: 7
- IfcPipeSegment: 5
- IfcCivilElement: 3
- IfcCableCarrierFitting: 2
- IfcElectricAppliance: 2
- IfcRampFlight: 2
- IfcShadingDevice: 1
- IfcFlowSegment: 1
